# 03 — HuggingFace Optimum (ORT backend)

Loads the ONNX model (produced in notebook 02) via HuggingFace Optimum's
`ORTModelForImageClassification`. This tests the Optimum ORT wrapper overhead
and compatibility vs. raw OnnxRuntime.

**Prerequisites:** `models/model.onnx` must exist (run notebook 02 first).

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from optimum.onnxruntime import ORTModelForImageClassification

from src.preprocess import load_and_preprocess

ONNX_PATH = Path('../models/model.onnx')
# Optimum expects a directory with model.onnx inside
OPTIMUM_MODEL_DIR = Path('../models/optimum')
DATASET_CSV = Path('../data/dataset.csv')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
# Optimum loads from a directory containing model.onnx
OPTIMUM_MODEL_DIR.mkdir(parents=True, exist_ok=True)
dest = OPTIMUM_MODEL_DIR / 'model.onnx'
if not dest.exists():
    shutil.copy(ONNX_PATH, dest)
    print(f'Copied {ONNX_PATH} -> {dest}')
else:
    print(f'Already exists: {dest}')

In [ ]:
# Load via Optimum ORT backend
# num_labels=4 for 0/90/180/270 orientation classes
model = ORTModelForImageClassification.from_pretrained(
    str(OPTIMUM_MODEL_DIR),
    local_files_only=True,
)
print('Model loaded via Optimum ORT')
print(f'Providers: {model.model.get_providers()}')

In [ ]:
import torch

def predict(image_path: str) -> tuple[int, float]:
    # load_and_preprocess returns (1, 3, 224, 224) numpy float32
    tensor_np = load_and_preprocess(image_path)
    pixel_values = torch.from_numpy(tensor_np)  # Optimum expects torch tensor
    outputs = model(pixel_values=pixel_values)
    logits = outputs.logits.detach().numpy()
    pred_label = int(np.argmax(logits, axis=1)[0])
    confidence = float(np.max(logits))
    return pred_label, confidence

df = pd.read_csv(DATASET_CSV)

predictions, confidences, latencies = [], [], []

for row in tqdm(df.itertuples(), total=len(df), desc='Optimum ORT inference'):
    t0 = time.perf_counter()
    pred, conf = predict(row.image_path)
    latencies.append((time.perf_counter() - t0) * 1000)
    predictions.append(pred)
    confidences.append(conf)

df['optimum_pred'] = predictions
df['optimum_conf'] = confidences
df['optimum_latency_ms'] = latencies
df.to_csv(RESULTS_DIR / 'optimum_results.csv', index=False)
print('Saved results/optimum_results.csv')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

accuracy = (df['optimum_pred'] == df['label']).mean()
avg_latency = df['optimum_latency_ms'].mean()
throughput = 1000 / avg_latency

print(f'Accuracy:         {accuracy:.4f}')
print(f'Avg latency:      {avg_latency:.2f} ms')
print(f'Throughput:       {throughput:.1f} img/s')
print()
print(classification_report(df['label'], df['optimum_pred'],
                             target_names=['0deg','90deg','180deg','270deg']))

cm = confusion_matrix(df['label'], df['optimum_pred'])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', ax=ax,
            xticklabels=['0','90','180','270'],
            yticklabels=['0','90','180','270'])
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Optimum ORT — Confusion Matrix')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'optimum_confusion_matrix.png', dpi=150)
plt.show()